# Sampling frame: from 487 to 149 repositories

The annotation sample (RQ1 and RQ2) was drawn from a frame of 149 repositories, obtained in May 2026
from the 487 repositories with two or more files:

1. **Size (P35).** We kept the file pairs in which both files have more than 92.95 characters, the
   35th percentile of the file size in these repositories: 262 pairs in 205 repositories.
2. **Similarity.** We kept the pairs with textual similarity of at most 0.5
   (`difflib.SequenceMatcher(None, a, b).ratio()`), leaving out near-identical copies:
   185 pairs in 149 repositories (`data/rq02/rq02_frame_149.csv`).

This computation used the content read live from GitHub in May 2026, which was not stored. The
frame of 149 is therefore used as frozen data, and this notebook shows the similarity criterion with
the files downloaded in June 2026 (`data/collection/context_files/`). Small differences are expected
for files edited between the two dates.

In [1]:
import difflib
import itertools
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data" / "collection").exists():
    ROOT = ROOT.parent

FILES = ROOT / "data/collection/context_files"
POPULATION = ROOT / "data/rq02/rq02_population_487.csv"
FRAME_149 = ROOT / "data/rq02/rq02_frame_149.csv"
OUT = ROOT / "results/rq01"
OUT.mkdir(parents=True, exist_ok=True)

P35 = 92.95
MAX_SIMILARITY = 0.5
NAME = {"COPILOT": "copilot-instructions.md", "CURSOR": ".cursorrules"}
folders = {p.name.lower(): p for p in FILES.iterdir() if p.is_dir()}


def read(repo, label):
    folder = folders.get(repo.strip().lower().replace("/", "__"))
    target = NAME.get(label.strip().upper(), label.strip()).lower()
    found = [f for f in folder.iterdir() if f.name.lower() == target] if folder else []
    return found[0].read_text(encoding="utf-8", errors="ignore") if found else None

In [2]:
frame = pd.read_csv(FRAME_149)
frame_pairs = {(r.repo.strip().lower(), frozenset([r.file1.strip().upper(), r.file2.strip().upper()]))
               for r in frame.itertuples()}
print(f"Frame: {len(frame_pairs)} pairs in {frame['repo'].nunique()} repositories")

pairs = []
for r in pd.read_csv(POPULATION).itertuples():
    labels = [x.strip() for x in r.arquivos_para_analisar.split(";") if x.strip()]
    for x, y in itertools.combinations(labels, 2):
        a, b = read(r.repo, x), read(r.repo, y)
        if a is None or b is None or len(a) <= P35 or len(b) <= P35:
            continue
        pairs.append({"repo": r.repo, "file1": x, "file2": y,
                      "similarity": round(difflib.SequenceMatcher(None, a, b).ratio(), 4),
                      "in_frame_149": (r.repo.lower(), frozenset([x.upper(), y.upper()])) in frame_pairs})
pairs = pd.DataFrame(pairs)
pairs.to_csv(OUT / "pair_similarity.csv", index=False)

summary = pairs.groupby("in_frame_149")["similarity"].agg(
    pairs="size", median="median",
    above_05=lambda s: int((s > MAX_SIMILARITY).sum()), identical=lambda s: int((s == 1.0).sum()))
summary

Frame: 185 pairs in 149 repositories


,pairs,median,above_05,identical
in_frame_149,,,,
False,353,1.0000,259,232
True,179,0.0309,2,1


Reading: the pairs kept in the frame have low similarity and almost none is above 0.5; most of
those left out are identical copies.